# Add IIIF facsimiles and measure zones to several MEI files

**Workflow 1 — handle MEI files.** This is the batch form of [`mei_single_file_iiif_integration.ipynb`](mei_single_file_iiif_integration.ipynb). List each page as a job, review the plan, then run the same steps for every file: copy to a working name, download the facsimile picture, detect measure boxes with the Edirom neural-network detector, and write `{stem}_facs_zones.mei`.

The example uses three clean Buxtehude pages in `test_corpus/` (no `<facsimile>` yet). Generated files go to `converted_mei/iiif_batch/` (gitignored). The originals are not overwritten.

If you already have a folder of `bsb00023199_00175.mei`-style files, you can replace the `JOBS` list with `collect_iiif_jobs_from_directory(...)`.

Network calls and writes stay off until you set `RUN_IIIF_INTEGRATION = True`. Continue afterwards with [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb).

In [2]:
# Cloud Jupyter: install CAMAT and copy notebooks/test_corpus if they are not already here.
try:
    import setup_camat
except ModuleNotFoundError:
    pass
try:
    from camat.notebook_workspace import prepare_notebook
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "camat"])
    from camat.notebook_workspace import prepare_notebook

prepare_notebook()

# You can leave this cell unchanged.
try:
    from camat import (
        DETECTOR_URL,
        collect_iiif_jobs_from_directory,
        format_iiif_page_plans,
        format_iiif_page_results,
        integrate_iiif_pages,
        plan_iiif_pages,
    )
except ModuleNotFoundError:
    import setup_camat
    from camat import (
        DETECTOR_URL,
        collect_iiif_jobs_from_directory,
        format_iiif_page_plans,
        format_iiif_page_results,
        integrate_iiif_pages,
        plan_iiif_pages,
    )

## 1. Configure the jobs

Each row is one MEI page. Identify its facsimile in either or both of these ways:

- **`archive_url`** — a Digitale Sammlungen viewer link. CAMAT names the working copy from the BSB id and page, for example `bsb00023199_00175.mei`.
- **`iiif_image_url`** — a full IIIF image URL. When set, that address is downloaded instead of a URL built from the working filename.

If both are set, `archive_url` still controls the working filename and `iiif_image_url` controls the image. If the file is already named `bsb…_NNNNN.mei`, both may be left empty.

The **measure detector** is Edirom's DOMD service: a neural-network model that finds measure boxes on the facsimile *picture*. It does not read the MEI.

Review the plan, then set `RUN_IIIF_INTEGRATION = True`.

In [3]:
# One dict per page. Add or remove rows as needed.
JOBS = [
    {
        "source_mei": "test_corpus/Buxtehude-Anhang-S._175_musicxml_verovio.mei",
        "archive_url": "https://digitale-sammlungen.de/en/view/bsb00023199?page=175",
        "iiif_image_url": "",
    },
    {
        "source_mei": "test_corpus/Buxtehude-Anhang-S._178_musicxml_verovio.mei",
        "archive_url": "https://digitale-sammlungen.de/en/view/bsb00023199?page=178",
        "iiif_image_url": "",
    },
    {
        "source_mei": "test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei",
        "archive_url": "https://digitale-sammlungen.de/en/view/bsb00023199?page=185",
        "iiif_image_url": "",
    },
]

# If you already have a folder of bsb…_NNNNN.mei files, you can replace JOBS with:
# JOBS = collect_iiif_jobs_from_directory("path/to/page_mei_folder")

TARGET_DIR = "converted_mei/iiif_batch"

# Optional page filters, using the archive/filename page number. Examples: "175,185" or "175-".
PAGES = None
SKIP_PAGES = None

TARGET_DPI = 500
PAGE_WIDTH_MM = 210.0
MINIMUM_MEASURES = 5
MAX_MEASURE_MISMATCH = 1
TIMEOUT = 180

# Network calls, detector upload, and all writes stay off until this is True.
RUN_IIIF_INTEGRATION = False

OVERWRITE_STAGED_MEI = True
OVERWRITE_IMAGES = True
OVERWRITE_OUTPUT = True
REUSE_ANNOTATIONS = True

## 2. Review the plan

This cell only reads the sources and prints the working names. It does not download or change files.

In [4]:
plans = plan_iiif_pages(
    JOBS,
    target_dir=TARGET_DIR,
    pages=PAGES,
    skip_pages=SKIP_PAGES,
)
print(format_iiif_page_plans(plans))
print(f"Target dir: {TARGET_DIR}")
print(f"Detector:   {DETECTOR_URL}")
print(f"Run enabled: {RUN_IIIF_INTEGRATION}")

 page  stem                    source                                                    clean
-----  ----------------------  --------------------------------------------------------  -----
  175  bsb00023199_00175       test_corpus/Buxtehude-Anhang-S._175_musicxml_verovio.mei  yes
  178  bsb00023199_00178       test_corpus/Buxtehude-Anhang-S._178_musicxml_verovio.mei  yes
  185  bsb00023199_00185       test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei  yes
3 job(s)
Target dir: converted_mei/iiif_batch
Detector:   https://measure-detector.edirom.de/upload
Run enabled: True


## 3. Run the batch

With the flag enabled, each planned job is processed in order. A failure on one page is recorded and the remaining pages still run. Set `REUSE_ANNOTATIONS = True` to skip another detector upload when the annotation XML already exists.

In [5]:
if not RUN_IIIF_INTEGRATION:
    print("Skipped. Review the plan, then set RUN_IIIF_INTEGRATION = True.")
else:
    results = integrate_iiif_pages(
        plans,
        target_dpi=TARGET_DPI,
        page_width_mm=PAGE_WIDTH_MM,
        timeout=TIMEOUT,
        minimum_measures=MINIMUM_MEASURES,
        max_measure_mismatch=MAX_MEASURE_MISMATCH,
        overwrite_staged=OVERWRITE_STAGED_MEI,
        overwrite_images=OVERWRITE_IMAGES,
        overwrite_output=OVERWRITE_OUTPUT,
        reuse_annotations=REUSE_ANNOTATIONS,
        continue_on_error=True,
    )
    print(format_iiif_page_results(results))

  uploading image: bsb00023199_00175.jpg
  wrote annotations: bsb00023199_00175_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00175_facs_zones.mei
  reusing annotations: bsb00023199_00175_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00175_facs_zones.mei
  uploading image: bsb00023199_00178.jpg
  wrote annotations: bsb00023199_00178_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00178_facs_zones.mei
  reusing annotations: bsb00023199_00178_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00178_facs_zones.mei
  uploading image: bsb00023199_00185.jpg
  wrote annotations: bsb00023199_00185_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00185_facs_zones.mei
  reusing annotations: bsb00023199_00185_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00185_facs_zones.mei
3/3 job(s) succeeded
  OK   bsb00023199_00175: converted_mei/iiif_batch/bsb00023199_00175_facs_zones.mei
  OK   bsb00023199_00178: converted_mei/iiif_b

## Notes for the next batch

1. Change the `JOBS` list, or point `collect_iiif_jobs_from_directory` at a folder of `bsb…_NNNNN.mei` files.
2. Review the plan, then set `RUN_IIIF_INTEGRATION = True`.
3. Use `PAGES` / `SKIP_PAGES` to run a subset, for example `"175,185"` or `"1-29"` for preface pages to skip.
4. If a staged working copy already exists and should be replaced, set `OVERWRITE_STAGED_MEI = True`.
5. To rerun measure detection, set `REUSE_ANNOTATIONS = False` and `OVERWRITE_OUTPUT = True`.
6. Inspect a result in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb). For a volume that is already named for BSB stems, the maintainer notebook [`run_pipeline_workflow.ipynb`](../run_pipeline_workflow.ipynb) wraps `camat-run-pipeline`.